In [1]:
#!pip install python-binance 

In [2]:
#IMPORTS
import pandas as pd
import numpy as np
import time
import math
import os.path

from tqdm import tqdm
from datetime import timedelta, datetime
from dateutil import parser
from binance.enums import HistoricalKlinesType

In [ ]:
import json
from binance.client import Client


##CONSTANTS
binsizes = {"1m":1,"5m":5,"15m":15,"1h":60,"1d":1440}
binance_client = Client()
data_folder = "/Users/chinjieheng/Documents/data/binance_1Hdata/"

Create a data folder to store all raw csv files downloaded from Binance. 
These raw files are important to determine what data is already downloaded and hence need to download from Binance again. 
Downloading full data from Binance may take a long time.
Use vpn, in M'sia can't connect
Set the path name for the data folder in data_folder

Functions adapted from Peter Nistrup's write up

In [ ]:
def minutes_of_new_data(symbol, kline_size, data, source):
    """Fixed logic errors in timestamp handling"""
    
    if len(data) > 0:  
        # ERROR 1: Need to handle indexed DataFrame properly
        if isinstance(data.index, pd.DatetimeIndex):
            # Data has timestamp as index (after set_index)
            old = data.index[-1]
        elif 'timestamp' in data.columns:
            # Data has timestamp as column
            old = parser.parse(str(data["timestamp"].iloc[-1]))
        else:
            # Fallback if no timestamp found
            old = datetime.strptime('1 Jan 2022', '%d %b %Y')
    elif source == "binance": 
        old = datetime.strptime('1 Jan 2022', '%d %b %Y')
    else:
        # ERROR 2: No fallback for non-binance sources
        old = datetime.now()

    if source == "binance": 
        # ERROR 3: Should handle API errors
        try:
            new = pd.to_datetime(binance_client.get_klines(symbol=symbol, interval=kline_size)[-1][0], unit='ms')
        except Exception as e:
            print(f"Error getting latest timestamp: {e}")
            new = datetime.now()
    else:
        # ERROR 4: No handling for non-binance sources
        new = datetime.now()

    return old, new

def get_all_binance(symbol, kline_size, save = False):
    try:
            
        filename = data_folder + '%s-%s-data.parquet' % (symbol, kline_size)
        
        if os.path.isfile(filename): 
            data_df = pd.read_parquet(filename)
            # FIX: Remove 'ignore' column from existing data if it exists
            if 'ignore' in data_df.columns:
                data_df = data_df.drop('ignore', axis=1)
            # FIX: Convert existing data columns to numeric (they might be strings)
            numeric_columns = ['open', 'high', 'low', 'close', 'volume', 'quote_av', 'trades', 'tb_base_av', 'tb_quote_av']
            for col in numeric_columns:
                if col in data_df.columns:
                    data_df[col] = pd.to_numeric(data_df[col], errors='coerce')
        else: 
            data_df = pd.DataFrame()
            
        oldest_point, newest_point = minutes_of_new_data(symbol, kline_size, data_df, source = "binance")
        delta_min = (newest_point - oldest_point).total_seconds()/60
        available_data = math.ceil(delta_min/binsizes[kline_size])
        
        if oldest_point == datetime.strptime('1 Jan 2022', '%d %b %Y'): 
            print('Downloading all available %s data for %s. Be patient..!' % 
                (kline_size, symbol))
        else: 
            print('Downloading %d minutes of new data available for %s, i.e. %d instances of %s data.' % 
                (delta_min, symbol, available_data, kline_size))
            
        klines = binance_client.get_historical_klines(symbol, kline_size, 
                                                    oldest_point.strftime("%d %b %Y %H:%M:%S"), 
                                                    newest_point.strftime("%d %b %Y %H:%M:%S"),
                                                    klines_type=HistoricalKlinesType.FUTURES)
        
        data = pd.DataFrame(klines, columns = ['timestamp', 'open', 'high', 'low', 'close', 
                                            'volume', 'close_time', 'quote_av', 'trades', 
                                            'tb_base_av', 'tb_quote_av', 'ignore' ])
        data['timestamp'] = pd.to_datetime(data['timestamp'], unit='ms')
        data.set_index('timestamp', inplace=True)
        # Convert numeric columns for new data
        numeric_columns = ['open', 'high', 'low', 'close', 'volume', 'quote_av', 'trades', 'tb_base_av', 'tb_quote_av']
        for col in numeric_columns:
            data[col] = pd.to_numeric(data[col], errors='coerce')
        if len(data_df) > 0:
            # Concatenate and remove duplicates
            combined_data = pd.concat([data_df, data])
            
            # ERROR 9: Remove duplicates based on timestamp
            combined_data = combined_data[~combined_data.index.duplicated(keep='last')]
            combined_data = combined_data.sort_index()
            data_df = combined_data
        else: 
            data_df = data
            

        
        if save: 
            data_df.to_parquet(filename)
            
        print('All caught up..!')
        return data_df
    
    except Exception as e:
        print(f'ERROR in {symbol}: {str(e)}')
        # Don't re-raise - just return existing data or empty DataFrame
        # This allows delisted/invalid symbols to be skipped gracefully
        if os.path.isfile(data_folder + '%s-%s-data.parquet' % (symbol, kline_size)):
            return pd.read_parquet(data_folder + '%s-%s-data.parquet' % (symbol, kline_size))
        return pd.DataFrame()

data["timestamp"].iloc[-1]:

data["timestamp"] selects the timestamp column from the DataFrame data.
.iloc[-1] gets the last value in this column, which corresponds to the most recent timestamp in the existing data.
parser.parse):

This is a method from the dateutil library, specifically from the dateutil.parser module.
parser.parse() converts a string into a datetime 

datetime.strptime('1 Jan 2017', '%d %b %Y'):
datetime.strptime():
 A method from Python's datetime module that parses a string representing a date and/or time and converts it into a datetime object.
'1 Jan 2017': The string representing the date January 1, 2017.
'%d %b %Y': The format string that specifies how the date is represented in the str

binance_client.get_klines(symbol=symbol, interval=kline_size):

This function call retrieves Kline (candlestick) data for a specified cryptocurrency symbol and kline_size interval from the Binance API.
The result is a list of Kline data, where each Kline is represented as a list of values (timestamp, open, high, low, close, volume, etc.).
[-1][]:

[-1]: Accesses the last Kline (the most recent one) in the retrieved list of Klines.
[0]: Accesses the first element of the last Kline, which is the timestamp in milliseconds since the Unix epoch.
pd.to_datetime(..., unit'ms'):

pd.to_datetime(): A Pandas function that converts a given value to a datetime object.
unit='ms': Specifies that the input value is in milliseconds. The function will convert the timestamp from milliseconds to a datet

filename = data_folder + '%s-%s-data.csv' % (symbol, kline_size)
%s-%s-data.csv is the format string.
% (symbol, kline_size) is the tuple of variables to insert into the format string.
%s is a placeholder for a string.
Suppose symbol is 'BTCUSDT' and kline_size is '1h'. The result would y code
filename = data_folder + 'BTCUSDT-1h-data.csv'ime object.ingobject.

________________________________________________________________________________________________________________________________________________________

Get list of coins

Get a cryptocurrency symbols based in USD

In [5]:
info = binance_client.futures_exchange_info()

symbols = info['symbols']
coins = []
others = []

for i, symbol in enumerate(symbols):
    # Filter for perpetual contracts only
    if symbol['contractType'] != 'PERPETUAL':
        continue
    
    s = symbol['symbol']
    if ('USDT' in s):
#         print('{} - {}'.format(i, s))
        coins.append(s)

In [6]:
'''from tqdm import tnrange, notebook

for symbol in notebook.tqdm(coins):
    try:
        get_all_binance(symbol, '1m', save = True)
    except:
        print('Skipping {}...'.format(symbol))
        pass'''

for symbol in tqdm(coins):
    get_all_binance(symbol, '1h', save=True)
    time.sleep(0.5)  # Rate limit: 0.5 second delay between requests

  0%|          | 0/613 [00:00<?, ?it/s]

All caught up..!


  0%|          | 1/613 [00:00<08:06,  1.26it/s]

All caught up..!


  0%|          | 2/613 [00:01<08:06,  1.26it/s]

...

100%|█████████▉| 611/613 [07:47<00:01,  1.35it/s]

Error getting latest timestamp: APIError(code=-1121): Invalid symbol.
All caught up..!


100%|█████████▉| 612/613 [07:48<00:00,  1.37it/s]

Error getting latest timestamp: APIError(code=-1121): Invalid symbol.
All caught up..!


100%|██████████| 613/613 [07:48<00:00,  1.31it/s]


In my analysis, I am only interested in CLOSE price. Hence, I perform some data transformation and combined all the close prices into a single dataframe.

In [7]:
'''from functools import reduce


data_list = []
column_names = []

for symbol in notebook.tqdm(coins):
    
    file_path = data_folder + '%s-1m-data.csv' % (symbol)
    if os.path.isfile(file_path):
        try:
            data = pd.read_csv(data_folder + symbol + '-1m-data.csv', parse_dates=True, index_col='timestamp')
            data = pd.to_numeric(data['close']).resample('1T').last().rename(symbol[:11])
            data_list.append(data)
            column_names.append(symbol[:11])
        except Exception as e:
            print(f'Error processing {symbol}: {e}')
    else:
        print(f'File not found for {symbol}, skipping.')
    
# Check if data_list is not empty before attempting to merge
if data_list:
    prices = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True, how='outer'), data_list)
    
    # Ensure the number of columns matches the data_list
    prices.columns = column_names[:len(prices.columns)]'''

"from functools import reduce\n\n\ndata_list = []\ncolumn_names = []\n\nfor symbol in notebook.tqdm(coins):\n\n    file_path = data_folder + '%s-1m-data.csv' % (symbol)\n    if os.path.isfile(file_path):\n        try:\n            data = pd.read_csv(data_folder + symbol + '-1m-data.csv', parse_dates=True, index_col='timestamp')\n            data = pd.to_numeric(data['close']).resample('1T').last().rename(symbol[:11])\n            data_list.append(data)\n            column_names.append(symbol[:11])\n        except Exception as e:\n            print(f'Error processing {symbol}: {e}')\n    else:\n        print(f'File not found for {symbol}, skipping.')\n\n# Check if data_list is not empty before attempting to merge\nif data_list:\n    prices = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True, how='outer'), data_list)\n\n    # Ensure the number of columns matches the data_list\n    prices.columns = column_names[:len(prices.columns)]"

In [8]:
#prices.to_csv(data_folder + 'prices_backup.csv')

In [9]:
#prices = pd.read_csv(data_folder + 'prices_backup.csv', parse_dates=True, index_col='timestamp')
#prices.head()

In [10]:
#prices.tail()